# Módulo 1 — `KDMLayer` y clasificación

**Objetivo:** entender cómo un encoder + una KDM *aprendida* hacen clasificación — la Ecuación 12 del paper (inferencia con KDMs) y su traducción línea a línea en `KDMLayer.forward`.

**Requisito:** haber corrido el Módulo 0 (`00_fundamentos_matrices_densidad.ipynb`) — acá reusamos `pure2dm` y `dm2discrete` sin volver a explicarlos.

**Fuente matemática:** paper, §3.4 (densidades conjuntas, Ec. 9-11) y §3.5 (inferencia, Ec. 12; Algoritmo 3).
**Fuente de código:** `external/kdm/kdm/layers/kdm_layer.py`, `rbf_kernel_layer.py`, `cosine_kernel_layer.py`, `init.py`, `models/kdm_class_model.py`.

## 1. La densidad conjunta $\rho_{x,y}$ (§3.4)

En el Módulo 0 vimos KDMs para una sola variable. Para **clasificación** necesitamos modelar la relación entre una entrada $x$ y una salida $y$ — el paper lo hace con una **KDM conjunta**:

$$\rho_{x,y} = \big((x^{(\ell)}, y^{(\ell)})_{\ell=1\ldots m},\ (p_\ell)_{\ell=1\ldots m},\ k_X \otimes k_Y\big) \tag{Eq. 10}$$

Es una sola KDM con $m$ componentes, donde cada componente es un **par** $(x^{(\ell)}, y^{(\ell)})$, y el kernel es el producto de un kernel sobre $X$ (para las entradas) y uno sobre $Y$ (para las salidas). Esta $\rho_{x,y}$ es exactamente lo que `KDMLayer` **aprende** — sus componentes son parámetros entrenables, no datos fijos.

## 2. Inferencia: de una entrada a una distribución de salida (§3.5)

Dada una KDM de entrada $\rho_{x'} = ((x'^{(i)})_{i},\, (p'_i)_i,\, k_X)$ (una nueva entrada, posiblemente con incertidumbre) y la KDM conjunta aprendida $\rho_{x,y}$, la KDM de salida $\rho_y = ((y^{(\ell)})_\ell,\, (\pi_\ell)_\ell,\, k_Y)$ tiene los **mismos componentes $y^{(\ell)}$** que $\rho_{x,y}$, pero con probabilidades **recalculadas**:

$$\pi_\ell = \sum_{i=1}^{m'} p'_i\ \cdot\ \frac{p_\ell\, k_X\big(x^{(\ell)}, x'^{(i)}\big)^2}{\sum_{j=1}^{m} p_j\, k_X\big(x^{(j)}, x'^{(i)}\big)^2} \tag{Eq. 12}$$

El paper señala explícitamente el parecido con **Nadaraya-Watson kernel regression**: cada componente $\ell$ de la mezcla aprendida recibe un peso proporcional a qué tan *parecida* es la entrada nueva $x'^{(i)}$ a su propio $x^{(\ell)}$ (medido por el kernel), normalizado entre todos los componentes. La diferencia con NWKR clásico: en vez de devolver solo un punto estimado, devolvemos una **distribución completa** sobre $y$ (una KDM).

## 3. La Ec. 12, línea por línea en `KDMLayer`

```python
# external/kdm/kdm/layers/kdm_layer.py:33-58 — el __init__
c_x = torch.randn(n_comp, dim_x) * 0.05      # x^(l), l=1..m  (n_comp = m)
c_y = torch.full((n_comp, dim_y), sqrt(1/dim_y))  # y^(l)
c_w = torch.full((n_comp,), 1.0 / n_comp)         # p_l (antes de normalizar)
self.c_x = nn.Parameter(c_x, requires_grad=x_train)
self.c_y = nn.Parameter(c_y, requires_grad=y_train)
self.c_w = nn.Parameter(c_w, requires_grad=w_train)
```

`c_x`, `c_y`, `c_w` **son** $\{x^{(\ell)}\}$, $\{y^{(\ell)}\}$, $\{p_\ell\}$ de la Ec. 10 — parámetros de `nn.Module`, entrenables por gradiente.

```python
# kdm_layer.py:64-85 — _compute_mixture + forward
def _compute_mixture(self, rho_in):
    in_w = rho_in[:, :, 0]            # p'_i   (pesos de la KDM de entrada)
    in_v = rho_in[:, :, 1:]           # x'^(i) (vectores de la KDM de entrada)
    comp_w = self._normalized_comp_w()          # p_l, normalizados a sumar 1
    out_vw = self.kernel(in_v, self.c_x)         # k_X(x'^(i), x^(l))  -> (bs, m', m)
    out_w = comp_w.view(1,1,-1) * out_vw.square()  # p_l * k_X(...)^2  <- numerador de Eq. 12
    return in_w, out_w

def forward(self, rho_in):
    in_w, out_w = self._compute_mixture(rho_in)
    out_w = out_w.clamp(min=self.eps)
    out_w = out_w / out_w.sum(dim=2, keepdim=True)   # <- normaliza sobre l: la FRACCION completa de Eq. 12
    out_w = torch.einsum('...i,...ij->...j', in_w, out_w)  # <- suma ponderada por p'_i: pi_l
    out_w = out_w.unsqueeze(-1)
    out_y = self.c_y.unsqueeze(0).expand(out_w.shape[0], -1, -1)
    return torch.cat((out_w, out_y), dim=2)          # KDM de salida: (pi_l, y^(l))
```

Cada línea corresponde a una pieza de la Ec. 12 — la línea `out_w / out_w.sum(dim=2,...)` es exactamente la fracción (peso relativo de cada componente $\ell$ dado el punto de consulta $i$), y el `einsum` siguiente es la suma ponderada por $p'_i$ que agrega sobre todas las posibles entradas $i$. El resultado final se empaqueta como una KDM nueva `(pi_l, y^(l))` — **la misma representación tensorial `(bs,n,d+1)` del Módulo 0**, lista para encadenarse con otra capa o convertirse en distribución categórica vía `dm2discrete`.

## 4. Los kernels: RBF (continuo) y coseno (discreto)

```python
# rbf_kernel_layer.py:57-70
def forward(self, A, B):          # A: (bs,n,d)  B: (m,d)
    AB = torch.matmul(A, B.transpose(-1,-2))
    dist2 = (A_norm + B_norm - 2*AB).clamp(min=0.)
    return torch.exp(-dist2 / (2*self.sigma**2))     # k_rbf,sigma(x,y) del Modulo 0 (Sec. 4)
```

```python
# cosine_kernel_layer.py:10-20
def forward(self, A, B):
    A = F.normalize(A, p=2, dim=-1)
    B = F.normalize(B, p=2, dim=-1)
    return torch.einsum("...nd,md->...nm", A, B)     # k_cos(x,y) del Modulo 0 (Sec. 3)
```

**En un `KDMClassModel`**: el kernel sobre $X$ (entradas, espacio continuo del encoder) es **RBF**; el kernel sobre $Y$ (etiquetas, categóricas) se maneja implícitamente porque `c_y` se inicializa/entrena como vectores tipo one-hot y se lee al final con `dm2discrete` — que ya vimos en el Módulo 0 que corresponde exactamente al kernel coseno (Ec. 5).

Nota sobre `sigma`: no es un simple float — se parametriza como `softplus(raw_sigma) + min_sigma` (línea 44) para garantizar que sea positivo durante todo el entrenamiento por gradiente, sin necesidad de recortarlo manualmente en cada paso.

## 5. Inicialización guiada por datos: `init_kdm_layer`

Antes de entrenar por gradiente, conviene que `c_x`/`c_y` empiecen en valores razonables — no aleatorios. `init_kdm_layer` (`kdm/init.py:25-51`) sobreescribe los componentes con una muestra real de datos, y opcionalmente fija `sigma` con una heurística de vecinos más cercanos:

```python
def _sigma_from_knn(points, sigma_mult=1.0):
    nn_model = NearestNeighbors(n_neighbors=3)
    nn_model.fit(points)
    distances, _ = nn_model.kneighbors(points)
    return float(np.mean(distances[:, 2]) * sigma_mult)   # distancia media al 3er vecino

def init_kdm_layer(layer, encoded_x, samples_y, init_sigma=False, sigma_mult=1.0):
    if init_sigma:
        layer.kernel.sigma = _sigma_from_knn(encoded_x.numpy(), sigma_mult)
    layer.c_x.copy_(encoded_x)     # componentes x^(l) = una muestra real, ya codificada
    layer.c_y.copy_(samples_y)     # componentes y^(l) = sus etiquetas correspondientes
    layer.c_w.fill_(1.0 / layer.n_comp)
```

**Requisito importante**: `n_comp` de la capa debe ser exactamente igual al número de filas de `encoded_x`/`samples_y` que se le pasan — la inicialización es una copia directa, no un muestreo aleatorio de tamaño distinto.

## 6. Ejercicio práctico: clasificar dígitos de MNIST

Construimos el pipeline completo — encoder pequeño (MLP, no CNN, para que corra rápido en CPU) + `RBFKernelLayer` + `KDMLayer`, exactamente lo que hace `KDMClassModel` (`kdm/models/kdm_class_model.py:7-58`):

```python
def forward(self, x):
    encoded = self.encoder(x)      # phi(x)
    rho_x = pure2dm(encoded)       # KDM de UN componente -- "sin incertidumbre" (Algoritmo 3, paso 4)
    rho_y = self.kdm(rho_x)        # Ec. 12
    return dm2discrete(rho_y)      # Ec. 5/15 -- distribucion categorica final
```

Es la cadena completa de los dos módulos: `pure2dm` (Módulo 0) → `KDMLayer.forward` (Ec. 12, arriba) → `dm2discrete` (Módulo 0, Ec. 5).

In [ ]:
import torch
import torch.nn as nn
import torchvision

from kdm.models import KDMClassModel
from kdm.init import init_kdm_layer

torch.manual_seed(42)

# --- Datos: subconjunto chico de MNIST (velocidad, no SOTA) ---
mnist_train = torchvision.datasets.MNIST(root="../../../data/mnist", train=True, download=False)
mnist_test = torchvision.datasets.MNIST(root="../../../data/mnist", train=False, download=False)

N_TRAIN, N_TEST = 3000, 1000
X_train = mnist_train.data[:N_TRAIN].float().reshape(N_TRAIN, -1) / 255.0   # (N, 784)
y_train = mnist_train.targets[:N_TRAIN]
X_test = mnist_test.data[:N_TEST].float().reshape(N_TEST, -1) / 255.0
y_test = mnist_test.targets[:N_TEST]

print("X_train:", X_train.shape, " X_test:", X_test.shape)

In [ ]:
# --- Encoder pequeno: MLP 784 -> 32 (no CNN, para que corra rapido en CPU) ---
ENCODED_SIZE = 32
encoder = nn.Sequential(
    nn.Linear(784, 64), nn.ReLU(),
    nn.Linear(64, ENCODED_SIZE),
)

N_COMP = 100   # 10 componentes por digito -- suficiente para el ejercicio, corre rapido
model = KDMClassModel(
    encoded_size=ENCODED_SIZE, dim_y=10, encoder=encoder,
    n_comp=N_COMP, sigma=1.0, sigma_trainable=True,
)

# --- init_kdm_layer: componentes iniciales = una muestra REAL ya codificada (Sec. 5 arriba) ---
idx_init = torch.randperm(N_TRAIN)[:N_COMP]
x_init = X_train[idx_init]
y_init_onehot = torch.nn.functional.one_hot(y_train[idx_init], num_classes=10).float()
encoded_init = encoder(x_init).detach()
init_kdm_layer(model.kdm, encoded_init, y_init_onehot, init_sigma=True)

print("sigma inicial (heuristica k-NN):", model.kernel.sigma.item())
print("c_x:", model.kdm.c_x.shape, " c_y:", model.kdm.c_y.shape, " c_w:", model.kdm.c_w.shape)

In [ ]:
# --- Entrenamiento discriminativo (Algoritmo 3): cross-entropy sobre dm2discrete(rho_y) ---
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(15):
    perm = torch.randperm(N_TRAIN)
    total_loss = 0.0
    for i in range(0, N_TRAIN, 128):
        idx = perm[i:i+128]
        xb, yb = X_train[idx], y_train[idx]

        probs = model(xb)                                        # dm2discrete(rho_y), Ec. 5/15
        loss = torch.nn.functional.nll_loss(torch.log(probs + 1e-8), yb)

        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * len(idx)

    if epoch % 3 == 0 or epoch == 14:
        with torch.no_grad():
            preds = model(X_test).argmax(dim=1)
            acc = (preds == y_test).float().mean().item()
        print(f"epoca {epoch:2d}  loss={total_loss/N_TRAIN:.4f}  accuracy_test={acc:.3f}")

Con un encoder MLP chico, pocas épocas y solo 3000 muestras de entrenamiento, la exactitud no será de nivel SOTA (~99%) — el objetivo de este ejercicio es ver el **mecanismo de KDM funcionando y aprendiendo**, no competir con MNIST benchmarks. Reproducir esto con un encoder más grande (CNN) y más `n_comp`/datos es exactamente el patrón de `references/classification.md` en la skill de KDM.

In [ ]:
# --- Verificar que la salida de forward() es una distribucion categorica valida ---
with torch.no_grad():
    probs = model(X_test[:5])

print("Suma de probabilidades por muestra (debe ser ~1.0):", probs.sum(dim=1))
print("\nPrediccion vs. real (primeras 5 muestras de test):")
for i in range(5):
    print(f"  real={y_test[i].item()}  prediccion={probs[i].argmax().item()}  "
          f"prob_top1={probs[i].max().item():.3f}")

## Resumen y próximo módulo

- Una `KDMLayer` guarda una **KDM conjunta aprendida** $\rho_{x,y}$ en `c_x`, `c_y`, `c_w` — parámetros entrenables, no datos fijos.
- `forward()` implementa la Ec. 12 exactamente: cada componente $\ell$ recibe un peso proporcional a $k_X(x^{(\ell)}, x')^2$, normalizado — el mismo principio que Nadaraya-Watson, pero devolviendo una distribución completa (KDM), no un punto.
- `KDMClassModel.forward` es la cadena `pure2dm → KDMLayer.forward (Ec. 12) → dm2discrete (Ec. 5)` — conecta directamente los Módulos 0 y 1.
- `init_kdm_layer` arranca los componentes desde datos reales (no aleatorios) y fija `sigma` con una heurística de vecinos más cercanos, antes del ajuste fino por gradiente.

**Módulo 2** (`02_estimacion_de_densidad.ipynb`, pendiente): qué pasa cuando no hay "$y$" — `KDMProjLayer` y `KDMDenEstModel` para estimar la densidad de los datos en sí (§3.3 del paper), aplicado a la densidad de cada dígito de MNIST.